# **Neuromorphic computing - LAB 03**
May-June 2026, "Machine learning in applications" course

*Prof. G. Urgese, V. Fra, B. Leto*

---
Train a FFSNN using fMNIST dataset and explore population coding
---

For a comprehensive overview on how SNNs work, and what is going on under the hood, [then you might be interested in the snnTorch tutorial series available here.](https://snntorch.readthedocs.io/en/latest/tutorials/index.html)
The snnTorch tutorial series is based on the following paper. If you find these resources or code useful in your work, please consider citing the following source:

> <cite> [Jason K. Eshraghian, Max Ward, Emre Neftci, Xinxin Wang, Gregor Lenz, Girish Dwivedi, Mohammed Bennamoun, Doo Seok Jeong, and Wei D. Lu. "Training Spiking Neural Networks Using Lessons From Deep Learning". arXiv preprint arXiv:2109.12894, September 2021.](https://arxiv.org/abs/2109.12894) </cite>

In [ ]:
!pip install snntorch==0.6.2 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 65.3 MB/s eta 0:00:00


In [1]:
import torch, torch.nn as nn
import snntorch as snn

c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\ML in Application\LABS\Labs-FP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DataLoading
Define variables for dataloading.

In [2]:
batch_size = 128
data_path='/data/fmnist'
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [3]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define a transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

fmnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transform)
fmnist_test = datasets.FashionMNIST(data_path, train=False, download=True, transform=transform)

# Create DataLoaders
train_loader = DataLoader(fmnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(fmnist_test, batch_size=batch_size, shuffle=True)

100%|██████████| 26421880/26421880 [00:02<00:00, 11077188.19it/s]


Extracting /data/fmnist\FashionMNIST\raw\train-images-idx3-ubyte.gz to /data/fmnist\FashionMNIST\raw



100%|██████████| 29515/29515 [00:00<00:00, 461354.84it/s]


Extracting /data/fmnist\FashionMNIST\raw\train-labels-idx1-ubyte.gz to /data/fmnist\FashionMNIST\raw



100%|██████████| 4422102/4422102 [00:00<00:00, 5123276.78it/s]


Extracting /data/fmnist\FashionMNIST\raw\t10k-images-idx3-ubyte.gz to /data/fmnist\FashionMNIST\raw



100%|██████████| 5148/5148 [00:00<00:00, 5270265.31it/s]

Extracting /data/fmnist\FashionMNIST\raw\t10k-labels-idx1-ubyte.gz to /data/fmnist\FashionMNIST\raw



# Define Network
Let's compare the performance of a pair of networks both with and without population coding, and train them for *one single time step.*



In [4]:
from snntorch import surrogate

n_classes = len(train_loader.dataset.classes)

# network parameters
num_inputs = train_loader.dataset.data.shape[1]**2

num_hidden = 128
num_outputs = n_classes

# temporal dynamics
num_steps = 10

# spiking neuron parameters
beta = 0.9  # neuron decay rate
grad = surrogate.fast_sigmoid()

In [5]:
print(n_classes)

10


## Without population coding
Let's just use a simple 2-layer dense spiking network.

In [6]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(num_inputs, num_hidden),
                    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
                    nn.Linear(num_hidden, num_outputs),
                    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
                    ).to(device)

# With population coding

It is thought that rate codes alone cannot be the dominant encoding mechanism in the primary cortex. One of several reasons is because the average neuronal firing rate is roughly $0.1-1$ Hz, which is far slower than the reaction response time of animals and humans.

But if we pool together multiple neurons and count their spikes together, then it becomes possible to measure a firing rate for a population of neurons in a very short window of time. Population coding adds some credibility to the plausibility of rate-encoding mechanisms.

<center>
<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/examples/tutorial_pop/pop.png?raw=true' width="300">
</center>


In this tutorial, you will:
* Learn how to train a population coded network. Instead of assigning one neuron per class, we will extend this to multiple neurons per class, and aggregate their spikes together.

Instead of 10 output neurons corresponding to 10 output classes, we will use 500 output neurons. This means each output class has 50 neurons randomly assigned to it.

In [7]:

pop_outputs = n_classes * 50

net_pop = nn.Sequential(nn.Flatten(),
                        nn.Linear(num_inputs, num_hidden),
                        snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
                        nn.Linear(num_hidden, pop_outputs),
                        snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
                        ).to(device)

# Training

## Without population coding
Define the optimizer and loss function. Here, we use the MSE Count Loss, which counts up the total number of output spikes at the end of the simulation run.

The correct class has a target firing probability of 100%, and incorrect classes are set to 0%.

In [8]:
import snntorch.functional as SF

optimizer = torch.optim.Adam(net.parameters(), lr=2e-3, betas=(0.9, 0.999))
loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0)

We will also define a simple test accuracy function that predicts the correct class based on the neuron with the highest spike count.

In [9]:
from snntorch import utils

def test_accuracy(data_loader, net, num_steps, population_code=False, num_classes=False):
  with torch.no_grad():
    total = 0
    acc = 0
    net.eval()

    data_loader = iter(data_loader)
    for data, targets in data_loader:
      data = data.to(device)
      targets = targets.to(device)
      utils.reset(net)
      spk_rec, _ = net(data)

      if population_code:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets, population_code=True, num_classes=n_classes) * spk_rec.size(1)
      else:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets) * spk_rec.size(1)

      total += spk_rec.size(1)

  return acc/total

Let's run the training loop. Note that we are only training for $1$ time step. I.e., each neuron only has the opportunity to fire once. As a result, we might not expect the network to perform too well here.

In [10]:
from snntorch import backprop

num_epochs = 20

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net, train_loader, num_steps=num_steps,
                          optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net, num_steps)*100:.3f}%\n")

C:\Users\dorot\AppData\Local\Temp\ipykernel_41220\2440134412.py:1: DeprecationWarning: The module snntorch.backprop will be deprecated in  a future release. Writing out your own training loop will lead to substantially faster performance.
  from snntorch import backprop


Epoch: 0
Test set accuracy: 52.205%

Epoch: 1
Test set accuracy: 59.068%

Epoch: 2
Test set accuracy: 69.264%

Epoch: 3
Test set accuracy: 65.180%

Epoch: 4
Test set accuracy: 61.274%

Epoch: 5
Test set accuracy: 71.539%

Epoch: 6
Test set accuracy: 67.583%

Epoch: 7
Test set accuracy: 68.888%

Epoch: 8
Test set accuracy: 64.962%

Epoch: 9
Test set accuracy: 69.867%

Epoch: 10
Test set accuracy: 71.104%

Epoch: 11
Test set accuracy: 70.609%

Epoch: 12
Test set accuracy: 71.618%

Epoch: 13
Test set accuracy: 69.759%

Epoch: 14
Test set accuracy: 71.242%

Epoch: 15
Test set accuracy: 68.236%

Epoch: 16
Test set accuracy: 73.388%

Epoch: 17
Test set accuracy: 72.142%

Epoch: 18
Test set accuracy: 71.212%

Epoch: 19
Test set accuracy: 73.210%



While there are ways to improve single time-step performance, e.g., by applying the loss to the membrane potential, one single time-step is extremely challenging to train a network competitively using rate codes.

## With population coding
Let's modify the loss function to specify that population coding should be enabled. We must also specify the number of classes. This means that there will be a total of $50~neurons~per~class~=~500~neurons~/~10~classes$.

In [11]:
loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0, population_code=True, num_classes=n_classes)
optimizer = torch.optim.Adam(net_pop.parameters(), lr=2e-3, betas=(0.9, 0.999))

In [12]:
num_epochs = 20

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net_pop, train_loader, num_steps=num_steps,
                            optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net_pop, num_steps, population_code=True, num_classes=n_classes)*100:.3f}%\n")

Epoch: 0
Test set accuracy: 79.302%

Epoch: 1
Test set accuracy: 82.694%

Epoch: 2
Test set accuracy: 81.833%

Epoch: 3
Test set accuracy: 84.098%

Epoch: 4
Test set accuracy: 82.664%

Epoch: 5
Test set accuracy: 80.429%

Epoch: 6
Test set accuracy: 84.721%

Epoch: 7
Test set accuracy: 84.949%

Epoch: 8
Test set accuracy: 84.098%

Epoch: 9
Test set accuracy: 84.563%

Epoch: 10
Test set accuracy: 83.653%

Epoch: 11
Test set accuracy: 84.790%

Epoch: 12
Test set accuracy: 85.166%

Epoch: 13
Test set accuracy: 84.177%

Epoch: 14
Test set accuracy: 84.009%

Epoch: 15
Test set accuracy: 84.405%

Epoch: 16
Test set accuracy: 84.721%

Epoch: 17
Test set accuracy: 85.156%

Epoch: 18
Test set accuracy: 85.631%

Epoch: 19
Test set accuracy: 84.286%



Even though we are only training on one time-step, introducing additional output neurons has immediately enabled better performance.

# Conclusion
The performance boost from population coding may start to fade as the number of time steps increases. But it may also be preferable to increasing time steps as PyTorch is optimized for handling matrix-vector products, rather than sequential, step-by-step operations over time.

* For a detailed tutorial of spiking neurons, neural nets, encoding, and training using neuromorphic datasets, check out the
[snnTorch tutorial series](https://snntorch.readthedocs.io/en/latest/tutorials/index.html).
* For more information on the features of snnTorch, check out the [documentation at this link](https://snntorch.readthedocs.io/en/latest/).
* If you have ideas, suggestions or would like to find ways to get involved, then [check out the snnTorch GitHub project here.](https://github.com/jeshraghian/snntorch)